In [ ]:
# Reference: Tomalak et al., Phys. Rev. D 106, 093006 (2022)
# https://journals.aps.org/prd/abstract/10.1103/PhysRevD.106.093006

# Also see https://arxiv.org/abs/0710.3897, showing that this should not explain the MiniBooNE excess

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import polars as pl
from tqdm.notebook import tqdm

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.rad_corrections import (
    alpha, m_e, m_mu, Delta_E,
    j_collinear, rad_frac_integrated_energy, rad_frac_x_eta, rad_frac_kinematic, bin_fraction_integrated,
)

from src.file_locations import intermediate_files_location


In [ ]:
# ── Shared parameters ─────────────────────────────────────────────────────────
Delta_theta_max_deg = 60.0            # angular range used in all plots [degrees]
E_nu = [200, 500, 1000, 2000]
colors     = ['royalblue', 'darkorange', 'forestgreen', 'crimson']
linestyles = ['-', '--', '-.', ':']
nu_labels  = [rf'$E_\mu^{{LO}} = {E}\,\mathrm{{MeV}}$' for E in E_nu]


# Reproduction Plot

In [ ]:
# Reproduction of Figure 4 of https://journals.aps.org/prd/abstract/10.1103/PhysRevD.106.093006

Delta_theta = np.linspace(0.0, 15.0, 3000)
energies   = [600, 2000, 6000]
fig4_ls    = [':',  '--', '-' ]
fig4_labs  = [r'$E_\ell + E_\gamma = 0.6\,\mathrm{GeV}$',
              r'$E_\ell + E_\gamma = 2\,\mathrm{GeV}$',
              r'$E_\ell + E_\gamma = 6\,\mathrm{GeV}$']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, m_ell, flav, ylim in [(axes[0], m_e, 'e', 0.20), (axes[1], m_mu, r'\mu', 0.05)]:
    for E_tree, ls, lab in zip(energies, fig4_ls, fig4_labs):
        ax.plot(Delta_theta, rad_frac_integrated_energy(Delta_theta, E_tree, m_ell),
                ls, color='k', linewidth=1.4, label=lab)
    ax.set_xlim(0, 15)
    ax.set_ylim(0, ylim)
    ax.set_xlabel(r'$\Delta\theta\,{}^\circ$', fontsize=13)
    ax.set_ylabel(r'$d\sigma^\gamma/d\sigma_{\rm LO}$', fontsize=13)
    ax.set_title(rf'static limit, ${flav}$ flavor', fontsize=12)
    ax.text(5.5, 0.06 * ylim, r'$E_\gamma < 20\,\mathrm{MeV}$', fontsize=10)
    ax.legend(fontsize=9, loc='lower right')
    ax.tick_params(direction='in', which='both')
fig.tight_layout()
plt.show()


# Gamma Distribution Plots

In [ ]:
# Example collinear photon energy and angle 1D distributions

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

energies = [200, 500, 1000, 2000]

# Left panel: photon energy spectrum  dσ^γ/(dE_γ dσ_LO)  [MeV⁻¹]
for E_tree, col, ls, lab in zip(energies, colors, linestyles, nu_labels):
    E_gamma_max = E_tree - m_mu
    if E_gamma_max <= Delta_E:
        continue
    E_gamma = np.linspace(Delta_E * 1.005, E_gamma_max * 0.998, 800)
    x        = 1.0 - E_gamma / E_tree
    eta_max  = np.deg2rad(Delta_theta_max_deg) * E_tree / m_mu
    spectrum = j_collinear(x, eta_max) / E_tree                      # MeV⁻¹
    ax1.plot(E_gamma, spectrum, ls, color=col, lw=1.5, label=lab)

ax1.set_xscale('log')
ax1.set_xlabel(r'$E_\gamma\;[\mathrm{MeV}]$', fontsize=13)
ax1.set_ylabel(r'$d\sigma^\gamma / (dE_\gamma\,d\sigma_{\rm LO})\;[\mathrm{MeV}^{-1}]$',
               fontsize=11)
ax1.set_title(rf'$\nu_\mu$ photon energy spectrum  ($\Delta\theta < {Delta_theta_max_deg:.0f}^\circ$)',
              fontsize=11)
ax1.set_ylim(bottom=0)
ax1.text(0.03, 0.97, r'$\Delta E = 20\,\mathrm{MeV}$',
         transform=ax1.transAxes, va='top', fontsize=10)
ax1.legend(fontsize=10)
ax1.tick_params(direction='in', which='both')

# Right panel: photon angle distribution  d(dσ^γ/dσ_LO)/d(Δθ°)
#   numerical derivative of Eq. (36) with respect to Δθ
Delta_theta_arr = np.linspace(0.05, Delta_theta_max_deg, 3000)
dth = 1e-4   # finite-difference step [degrees]

for E_tree, col, ls, lab in zip(energies, colors, linestyles, nu_labels):
    f_p = rad_frac_integrated_energy(Delta_theta_arr + dth, E_tree, m_mu)
    f_m = rad_frac_integrated_energy(Delta_theta_arr - dth, E_tree, m_mu)
    ax2.plot(Delta_theta_arr, (f_p - f_m) / (2.0 * dth), ls, color=col, lw=1.5, label=lab)

ax2.set_xlabel(r'$\Delta\theta\;[{}^\circ]$', fontsize=13)
ax2.set_ylabel(r'$(d\sigma^\gamma/d\sigma_{\rm LO})\,/\,d(\Delta\theta^\circ)$', fontsize=11)
ax2.set_title(r'$\nu_\mu$ photon angle distribution  ($E_\gamma > 20\,\mathrm{MeV}$)', fontsize=11)
ax2.set_xlim(0, Delta_theta_max_deg)
ax2.set_ylim(bottom=0)
ax2.text(0.97, 0.97, r'$\Delta E = 20\,\mathrm{MeV}$',
         transform=ax2.transAxes, va='top', ha='right', fontsize=10)
ax2.legend(fontsize=10)
ax2.tick_params(direction='in', which='both')

fig.tight_layout()
plt.show()

In [ ]:
# 2D distribution in x, eta
# This is what the later 3D (E_mu, E_gamma, DeltaTheta) distribution is derived from, 
# but we don't want to use this directly since we'd like to have a cutoff for low E_gamma, 
# and we would like to leave the muon energy distribution unaffected by the weighting.

fig = plt.figure(figsize=(12, 9))
ax = plt.gca()

x = np.linspace(0, 1, 100)
eta = np.linspace(0, 50, 100)
X, ETA = np.meshgrid(x, eta)

Z = rad_frac_x_eta(X, ETA)
vmax = float(Z.max())
vmin = vmax * 1e-3
im = ax.pcolormesh(X, ETA, Z, cmap='viridis', shading='gouraud',
                    norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))
levels = np.logspace(np.log10(vmin), np.log10(vmax), 7)

ax.contour(X, ETA, Z, levels=levels, colors='white', linewidths=0.6, alpha=0.4)

cbar = plt.colorbar(im, ax=ax, pad=0.02)
cbar.set_label(
    r'$d^2\sigma^\gamma\!/\!(dx d\eta\ d\sigma_{\rm LO})$', fontsize=8)

theta_char = np.rad2deg(m_mu / (E_tree))

ax.set_xlabel(r'x = $E_\mu/(E_\mu + E_\gamma)$', fontsize=11)
ax.set_ylabel(r'$\eta = \Delta\theta \cdot (E_\mu + E_\gamma) / m_\mu$', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 50)
ax.tick_params(direction='in', which='both')
plt.show()


In [ ]:
# Example 2D distribution of collinear photon energy and angle

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, E_tree in zip(axes.ravel(), energies):
    E_gamma_min = Delta_E * 1.01
    E_gamma_max = (E_tree - m_mu) * 0.999

    E_gamma = np.logspace(np.log10(E_gamma_min), np.log10(E_gamma_max), 500)
    theta   = np.linspace(0.2, Delta_theta_max_deg, 500)
    EE, TT  = np.meshgrid(E_gamma, theta)
    Z = np.clip(rad_frac_kinematic(EE, TT, E_tree), 0.0, None)

    vmax = float(Z.max())
    vmin = vmax * 1e-3
    im = ax.pcolormesh(EE, TT, Z, cmap='viridis', shading='gouraud',
                       norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))
    levels = np.logspace(np.log10(vmin), np.log10(vmax), 7)
    ax.contour(EE, TT, Z, levels=levels, colors='white', linewidths=0.6, alpha=0.4)

    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(
        r'$d^2\sigma^\gamma\!/\!(dE_\gamma\,d(\Delta\theta)^\circ\,d\sigma_{\rm LO})$'
        r'  [MeV$^{-1}$deg$^{-1}$]', fontsize=8)

    theta_char = np.rad2deg(m_mu / (E_tree))
    ax.axhline(theta_char, color='red', lw=1.2, ls='--', alpha=0.85)
    ax.text(E_gamma_min * 1.3, theta_char + 1.5,
            rf'$m_\mu/E_\mathrm{{tree}} = {theta_char:.1f}^\circ$', color='red', fontsize=8)

    ax.set_xscale('log')
    ax.set_xlabel(r'$E_\gamma\;[\mathrm{MeV}]$', fontsize=11)
    ax.set_ylabel(r'$\Delta\theta\;[{}^\circ]$', fontsize=11)
    ax.set_xlim(E_gamma_min, E_gamma_max)
    ax.set_ylim(0, Delta_theta_max_deg)
    ax.set_title(rf'$E_\mu^{{LO}} = {int(E_tree)}\,\mathrm{{MeV}}$', fontsize=12)
    ax.tick_params(direction='in', which='both')

fig.suptitle(
    r'$\nu_\mu$ 2D photon distribution  '
    r'$d^2\sigma^\gamma\!/\!(dE_\gamma\,d(\Delta\theta^\circ)\,d\sigma_{\rm LO})$'
    r'    [$\Delta E = 20\,\mathrm{MeV}$,  $\Delta\theta < 60^\circ$]',
    fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
# Example binned 2D distribution of collinear photon energy and angle

N_E     = 50   # E_gamma bins (log-spaced)
N_theta = 50   # Delta_theta bins (linear)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, E_tree in zip(axes.ravel(), E_nu):
    E_gamma_min = Delta_E
    E_gamma_max = E_tree - m_mu

    E_edges     = np.logspace(np.log10(E_gamma_min), np.log10(E_gamma_max), N_E + 1)
    theta_edges = np.linspace(0.0, Delta_theta_max_deg, N_theta + 1)
    E_centers     = np.sqrt(E_edges[:-1] * E_edges[1:])
    theta_centers = 0.5 * (theta_edges[:-1] + theta_edges[1:])

    EE, TT   = np.meshgrid(E_centers, theta_centers)
    dEE, dTT = np.meshgrid(np.diff(E_edges), np.diff(theta_edges))
    fractions = np.clip(rad_frac_kinematic(EE, TT, E_tree) * dEE * dTT, 0.0, None)

    total_pct_binned   = fractions.sum() * 100.0
    total_pct_analytic = rad_frac_integrated_energy(Delta_theta_max_deg, E_tree, m_mu) * 100.0

    im = ax.pcolormesh(E_edges, theta_edges, fractions,
                       cmap='plasma', vmin=0.0, vmax=fractions.max(), shading='flat')
    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(r'Fraction of $\sigma_{\rm LO}$ per bin', fontsize=9)

    theta_char = np.rad2deg(m_mu / E_tree)
    ax.axhline(theta_char, color='cyan', lw=1.2, ls='--', alpha=0.9)
    ax.text(E_gamma_min * 1.15, theta_char + 1.5,
            rf'$m_\mu/E_\mathrm{{tree}}={theta_char:.1f}^\circ$', color='cyan', fontsize=8)

    ax.set_xscale('log')
    ax.set_xlabel(r'$E_\gamma\;[\mathrm{MeV}]$', fontsize=11)
    ax.set_ylabel(r'$\Delta\theta\;[{}^\circ]$', fontsize=11)
    ax.set_xlim(E_gamma_min, E_gamma_max)
    ax.set_ylim(0.0, Delta_theta_max_deg)
    ax.set_title(
        rf'$E_\mu^{{LO}} = {int(E_tree)}\,\mathrm{{MeV}}$'
        rf'  —  binned: {total_pct_binned:.3f}%,  analytic: {total_pct_analytic:.3f}%',
        fontsize=10.5)
    ax.tick_params(direction='in', which='both')

fig.suptitle(
    r'$\nu_\mu$ photon fraction per $(E_\gamma,\,\Delta\theta)$ bin'
    r'   [$\Delta E=20\,\mathrm{MeV},\ \Delta\theta<60^\circ$,'
    rf'   {N_E}$\times${N_theta} bins]',
    fontsize=11)
fig.tight_layout()
plt.show()


In [ ]:
# Example coarse 3D binned distribution of collinear photon energy and angle

Etree_edges   = np.array([200, 300, 500, 750, 1000, 1250, 1500, 2000, 3000, 5000], dtype=float)
Etree_centers = 0.5 * (Etree_edges[:-1] + Etree_edges[1:])

N_E_coarse     = 6   # E_gamma bins per panel (log-spaced, per-panel kinematic range)
N_theta_coarse = 5   # Delta_theta bins per panel (linear, same for all panels)
# Total bins: 9 × 6 × 5 = 270

theta_edges_deg  = np.linspace(0.0, Delta_theta_max_deg, N_theta_coarse + 1)
theta_centers    = 0.5 * (theta_edges_deg[:-1] + theta_edges_deg[1:])

# Accumulate per-panel results for npz output
all_Egamma_edges = []   # (9, N_E_coarse+1)
all_weights      = []   # (9, N_theta_coarse, N_E_coarse)

fig, axes = plt.subplots(3, 3, figsize=(14, 11))

for ax, Etree_lo, Etree_hi, Etree_cen in zip(
        axes.ravel(),
        Etree_edges[:-1], Etree_edges[1:], Etree_centers):

    E_tree = Etree_cen                           # bin-center E_tree = E_μ^LO [MeV]
    E_gamma_min = Delta_E
    E_gamma_max = E_tree - m_mu             # kinematic max at bin center

    E_edges = np.logspace(np.log10(E_gamma_min), np.log10(E_gamma_max), N_E_coarse + 1)

    fractions = np.array([
        [bin_fraction_integrated(E_edges[j], E_edges[j+1],
                                 theta_edges_deg[k], theta_edges_deg[k+1],
                                 E_tree)
         for j in range(N_E_coarse)]
        for k in range(N_theta_coarse)
    ])

    all_Egamma_edges.append(E_edges)
    all_weights.append(fractions)

    total_pct      = fractions.sum() * 100.0
    analytic_pct   = rad_frac_integrated_energy(Delta_theta_max_deg, E_tree, m_mu) * 100.0

    im = ax.pcolormesh(E_edges, theta_edges_deg, fractions,
                       cmap='plasma', vmin=0.0, vmax=fractions.max(), shading='flat')
    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(r'Fraction of $\sigma_{\rm LO}$', fontsize=7)
    cbar.ax.tick_params(labelsize=7)

    theta_char = np.rad2deg(m_mu / E_tree)
    ax.axhline(theta_char, color='cyan', lw=1.0, ls='--', alpha=0.85)
    ax.text(E_gamma_min * 1.15, theta_char + 1.5,
            rf'$m_\mu/E_\mathrm{{tree}}={theta_char:.1f}^\circ$', color='cyan', fontsize=7)

    ax.set_xscale('log')
    ax.set_xlabel(r'$E_\gamma\;[\mathrm{MeV}]$', fontsize=9)
    ax.set_ylabel(r'$\Delta\theta\;[{}^\circ]$', fontsize=9)
    ax.set_xlim(E_gamma_min, E_gamma_max)
    ax.set_ylim(0.0, Delta_theta_max_deg)
    ax.set_title(
        rf'$E_\mu^{{LO}} \in [{int(Etree_lo)},{int(Etree_hi)}]\,\mathrm{{MeV}}$'
        rf'  —  {total_pct:.2f}% (analytic: {analytic_pct:.2f}%)',
        fontsize=8.5)
    ax.tick_params(direction='in', which='both', labelsize=8)

fig.suptitle(
    r'$\nu_\mu$ photon fraction per $(E_\gamma,\,\Delta\theta)$ bin'
    rf'  [$\Delta E=20\,\mathrm{{MeV}},\ \Delta\theta<60^\circ$,'
    rf'  {N_E_coarse}$\times${N_theta_coarse} bins/panel, 270 total]'
    r'  —  binned in $E_\mu^{LO} = E_\mu + E_\gamma$',
    fontsize=10)
fig.tight_layout()
plt.show()

# Save weights for MC reweighting
# weights[i_tree, i_theta, i_Egamma] = fraction of σ_LO in that 3D bin
# Look up by E_tree = E_mu + E_gamma (NOT E_nu); Δθ edges are shared across panels
np.savez('numu_3d_weights.npz',
         Etree_edges  = Etree_edges,
         Etree_centers = Etree_centers,
         theta_edges_deg  = theta_edges_deg,
         Egamma_edges = np.array(all_Egamma_edges),  # shape (9, N_E_coarse+1)
         weights          = np.array(all_weights),        # shape (9, N_theta_coarse, N_E_coarse)
)
print('Saved numu_3d_weights.npz')
print(f'  Bin layout: {len(Etree_edges)-1} E_tree × {N_theta_coarse} Δθ × {N_E_coarse} E_γ'
      f' = {(len(Etree_edges)-1) * N_theta_coarse * N_E_coarse} total bins')


# Del1g Plots

In [ ]:
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")

del1g_numuCC_df = all_df.filter(
    (pl.col("filetype") == "delete_one_gamma_overlay") & (pl.col("wc_truth_muonMomentum_3") > 0.0)
)
normal_numuCC_df = all_df.filter(
    (pl.col("filetype") != "delete_one_gamma_overlay") & (pl.col("wc_truth_muonMomentum_3") > 0.0)
)

relevant_vars = [
    "run",
    "subrun",
    "event",

    "wc_truth_muonMomentum_0",
    "wc_truth_muonMomentum_1",
    "wc_truth_muonMomentum_2",
    "wc_truth_muonMomentum_3",

    "wc_true_leading_shower_energy",
    "wc_true_leading_shower_costheta",
    "wc_true_leading_shower_phi",

    "wc_net_weight",
]

del1g_numuCC_df = del1g_numuCC_df.select(relevant_vars).collect()
normal_numuCC_df = normal_numuCC_df.select(relevant_vars).collect()

wc_truth_muonMomentum_0 = del1g_numuCC_df["wc_truth_muonMomentum_0"].to_numpy()
wc_truth_muonMomentum_1 = del1g_numuCC_df["wc_truth_muonMomentum_1"].to_numpy()
wc_truth_muonMomentum_2 = del1g_numuCC_df["wc_truth_muonMomentum_2"].to_numpy()
wc_truth_muonMomentum_3 = del1g_numuCC_df["wc_truth_muonMomentum_3"].to_numpy()

true_leading_shower_energies = del1g_numuCC_df["wc_true_leading_shower_energy"].to_numpy()
true_leading_shower_costhetas = del1g_numuCC_df["wc_true_leading_shower_costheta"].to_numpy()
true_leading_shower_phis = del1g_numuCC_df["wc_true_leading_shower_phi"].to_numpy() * np.pi / 180.0
true_leading_shower_0 = []
true_leading_shower_1 = []
true_leading_shower_2 = []
true_leading_shower_3 = []
true_muon_costheta = []
muon_gamma_opening_angles = []
rad_corr_E_tree = []
rad_corr_x = []
rad_corr_eta = []
rad_frac_x_eta_vals = []
for i in tqdm(range(len(true_leading_shower_energies))):
    costheta = true_leading_shower_costhetas[i]
    sintheta = np.sqrt(max(0.0, 1.0 - costheta*costheta))
    phi = true_leading_shower_phis[i]
    true_leading_shower_0.append(true_leading_shower_energies[i] * sintheta * np.cos(phi) / 1000.0)
    true_leading_shower_1.append(true_leading_shower_energies[i] * sintheta * np.sin(phi) / 1000.0)
    true_leading_shower_2.append(true_leading_shower_energies[i] * costheta / 1000.0)
    true_leading_shower_3.append(true_leading_shower_energies[i] / 1000.0)

    true_muon_costheta.append(wc_truth_muonMomentum_2[i] / wc_truth_muonMomentum_3[i])
    
    muon_dir_x = wc_truth_muonMomentum_0[i] / wc_truth_muonMomentum_3[i]
    muon_dir_y = wc_truth_muonMomentum_1[i] / wc_truth_muonMomentum_3[i]
    muon_dir_z = wc_truth_muonMomentum_2[i] / wc_truth_muonMomentum_3[i]
    gamma_dir_x = true_leading_shower_0[i] / true_leading_shower_3[i]
    gamma_dir_y = true_leading_shower_1[i] / true_leading_shower_3[i]
    gamma_dir_z = true_leading_shower_2[i] / true_leading_shower_3[i]
    dot_product = muon_dir_x * gamma_dir_x + muon_dir_y * gamma_dir_y + muon_dir_z * gamma_dir_z
    dot_product = min(max(dot_product, -1.0), 1.0)
    muon_gamma_opening_angles.append(np.arccos(dot_product) * 180.0 / np.pi)

    rad_corr_E_tree.append((wc_truth_muonMomentum_3[i] + true_leading_shower_3[i])*1000.0)
    rad_corr_x.append(wc_truth_muonMomentum_3[i]*1000.0 / rad_corr_E_tree[i])
    rad_corr_eta.append(np.arccos(costheta) * rad_corr_E_tree[i] / 105.658)
    
    rad_frac_x_eta_vals.append(rad_frac_x_eta(rad_corr_x[i], rad_corr_eta[i]))

del1g_numuCC_df = del1g_numuCC_df.with_columns(
    pl.Series(name="wc_true_leading_shower_0", values=true_leading_shower_0),
    pl.Series(name="wc_true_leading_shower_1", values=true_leading_shower_1),
    pl.Series(name="wc_true_leading_shower_2", values=true_leading_shower_2),
    pl.Series(name="wc_true_leading_shower_3", values=true_leading_shower_3),
    pl.Series(name="wc_true_muon_costheta", values=true_muon_costheta),
    pl.Series(name="wc_muon_gamma_opening_angle", values=muon_gamma_opening_angles),
    pl.Series(name="rad_corr_E_tree", values=rad_corr_E_tree),
    pl.Series(name="rad_corr_x", values=rad_corr_x),
    pl.Series(name="rad_corr_eta", values=rad_corr_eta),
    pl.Series(name="rad_frac_x_eta", values=rad_frac_x_eta_vals),
)

wc_truth_muonMomentum_0 = normal_numuCC_df["wc_truth_muonMomentum_0"].to_numpy()
wc_truth_muonMomentum_1 = normal_numuCC_df["wc_truth_muonMomentum_1"].to_numpy()
wc_truth_muonMomentum_2 = normal_numuCC_df["wc_truth_muonMomentum_2"].to_numpy()
wc_truth_muonMomentum_3 = normal_numuCC_df["wc_truth_muonMomentum_3"].to_numpy()
true_muon_costheta = []
for i in tqdm(range(len(wc_truth_muonMomentum_3))):
    true_muon_costheta.append(wc_truth_muonMomentum_2[i] / wc_truth_muonMomentum_3[i])
normal_numuCC_df = normal_numuCC_df.with_columns(
    pl.Series(name="wc_true_muon_costheta", values=true_muon_costheta),
)

del1g_numuCC_df = del1g_numuCC_df.filter(pl.col("wc_muon_gamma_opening_angle") < 60)

del1g_numuCC_df.select(
    "wc_truth_muonMomentum_3",
    "wc_true_leading_shower_3",
    "wc_true_muon_costheta",
    "rad_corr_E_tree",
    "rad_corr_x",
    "rad_corr_eta",
    "rad_frac_x_eta",
)


## 3D E_mu, x, eta Del1g plot

In [ ]:
# 3D E_mu, x, eta Del1g plot
# 9 panels (one per E_mu^LO bin), each panel is a 2D histogram of x vs eta,
# weighted by wc_net_weight

Etree_edges = np.array([0, 200, 500, 750, 1000, 1250, 1500, 2000, 3000, 5000], dtype=float)

N_x   = 10
N_eta = 10

rad_corr_E_tree_arr = np.array(del1g_numuCC_df["rad_corr_E_tree"])
rad_corr_x_arr      = np.array(del1g_numuCC_df["rad_corr_x"])
rad_corr_eta_arr    = np.array(del1g_numuCC_df["rad_corr_eta"])
weights_arr         = np.array(del1g_numuCC_df["wc_net_weight"])

fig, axes = plt.subplots(3, 3, figsize=(14, 11))

for ax, Etree_lo, Etree_hi in zip(axes.ravel(), Etree_edges[:-1], Etree_edges[1:]):
    mask = (rad_corr_E_tree_arr >= Etree_lo) & (rad_corr_E_tree_arr < Etree_hi)

    x_sel   = rad_corr_x_arr[mask]
    eta_sel = rad_corr_eta_arr[mask]
    w_sel   = weights_arr[mask]

    x_edges   = np.linspace(0, 1, N_x + 1)
    eta_edges = np.linspace(0, np.max(eta_sel), N_eta + 1)

    H, _, _ = np.histogram2d(x_sel, eta_sel, bins=[x_edges, eta_edges], weights=w_sel)

    vmax = H.max()
    vmin = vmax * 1e-3 if vmax > 0 else 1e-10
    H_plot = np.where(H > 0, H, np.nan)

    im = ax.pcolormesh(x_edges, eta_edges, H_plot.T, cmap='plasma', shading='flat',
                       norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))
    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(r'Weighted events / bin', fontsize=7)
    cbar.ax.tick_params(labelsize=7)

    n_events = mask.sum()
    ax.set_xlabel(r'$x = E_\mu / E_\mathrm{tree}$', fontsize=9)
    ax.set_ylabel(r'$\eta = \Delta\theta \cdot E_\mathrm{tree} / m_\mu$', fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, np.max(eta_edges))
    ax.set_title(
        rf'$E_\mu^{{LO}} \in [{int(Etree_lo)},{int(Etree_hi)}]\,\mathrm{{MeV}}$'
        rf'  —  {n_events:,} events',
        fontsize=8.5)
    ax.tick_params(direction='in', which='both', labelsize=8)

fig.suptitle(
    r'Del1g overlay: $\nu_\mu$CC events in $(x,\,\eta)$ plane'
    r'  binned by $E_\mu^{LO} = E_\mu + E_\gamma$  [weighted by $w_\mathrm{net}$]',
    fontsize=11)
fig.tight_layout()
plt.show()


In [ ]:
# Here, we add a new x_eta_uniform_weight column to the del1g_numuCC_df dataframe
# This value should make the weighted bin count uniform in each nonzero bin in each panel of the above plot.
# We'll re-make that plot to confirm this.

x_eta_uniform_weight = np.zeros(len(rad_corr_E_tree_arr))

for panel_idx in range(len(Etree_edges) - 1):
    Etree_lo, Etree_hi = Etree_edges[panel_idx], Etree_edges[panel_idx + 1]
    mask = (rad_corr_E_tree_arr >= Etree_lo) & (rad_corr_E_tree_arr < Etree_hi)
    if mask.sum() == 0:
        continue

    x_sel   = rad_corr_x_arr[mask]
    eta_sel = rad_corr_eta_arr[mask]
    w_sel   = weights_arr[mask]

    # Per-panel edges, matching the plot above exactly
    x_edges   = np.linspace(0, 1, N_x + 1)
    eta_edges = np.linspace(0, np.max(eta_sel), N_eta + 1)

    H, _, _ = np.histogram2d(x_sel, eta_sel, bins=[x_edges, eta_edges], weights=w_sel)

    # Digitize within this panel using its own eta_edges
    i_x   = np.clip(np.digitize(x_sel,   x_edges)   - 1, 0, N_x   - 1)
    i_eta = np.clip(np.digitize(eta_sel, eta_edges) - 1, 0, N_eta - 1)

    bin_vals = H[i_x, i_eta]
    uw = np.where(bin_vals > 0, 1.0 / bin_vals, 0.0)
    x_eta_uniform_weight[mask] = uw

del1g_numuCC_df = del1g_numuCC_df.with_columns(
    pl.Series("x_eta_uniform_weight", x_eta_uniform_weight)
)

# ── Validation plot: re-make the 3D plot with uniform weights ────────────────
combined_weights = weights_arr * x_eta_uniform_weight

fig, axes = plt.subplots(3, 3, figsize=(14, 11))

for ax, Etree_lo, Etree_hi in zip(axes.ravel(), Etree_edges[:-1], Etree_edges[1:]):
    mask = (rad_corr_E_tree_arr >= Etree_lo) & (rad_corr_E_tree_arr < Etree_hi)

    x_sel   = rad_corr_x_arr[mask]
    eta_sel = rad_corr_eta_arr[mask]

    x_edges   = np.linspace(0, 1, N_x + 1)
    eta_edges = np.linspace(0, np.max(eta_sel), N_eta + 1)

    H, _, _ = np.histogram2d(x_sel, eta_sel, bins=[x_edges, eta_edges],
                              weights=combined_weights[mask])
    H_plot = np.where(H > 0, H, np.nan)

    im = ax.pcolormesh(x_edges, eta_edges, H_plot.T, cmap='plasma', shading='flat',
                       vmin=0.9, vmax=1.1)
    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(r'Weighted events / bin', fontsize=7)
    cbar.ax.tick_params(labelsize=7)

    ax.set_xlabel(r'$x = E_\mu / E_\mathrm{tree}$', fontsize=9)
    ax.set_ylabel(r'$\eta = \Delta\theta \cdot E_\mathrm{tree} / m_\mu$', fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, np.max(eta_edges))
    ax.set_title(
        rf'$E_\mu^{{LO}} \in [{int(Etree_lo)},{int(Etree_hi)}]\,\mathrm{{MeV}}$',
        fontsize=8.5)
    ax.tick_params(direction='in', which='both', labelsize=8)

fig.suptitle(
    r'Validation: uniform-reweighted Del1g events in $(x,\,\eta)$ plane'
    r'  [each nonzero bin $\to$ 1]',
    fontsize=11)
fig.tight_layout()
plt.show()


## Energy plots

In [ ]:
plt.figure(figsize=(10, 6))
bins = np.linspace(0, 2000, 101)

plt.hist(np.array(del1g_numuCC_df["wc_truth_muonMomentum_3"])*1000.0 - 105.658, weights=del1g_numuCC_df["wc_net_weight"], bins=bins, histtype="step", label="Del1g overlay")
del1g_counts, bins, patches = plt.hist(np.array(del1g_numuCC_df["wc_truth_muonMomentum_3"])*1000.0 - 105.658, weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"], bins=bins, histtype="step", label="Del1g overlay (x-eta uniform weighted)")
normal_counts, bins, patches = plt.hist(np.array(normal_numuCC_df["wc_truth_muonMomentum_3"])*1000.0 - 105.658, weights=normal_numuCC_df["wc_net_weight"], bins=bins, histtype="step", label="normal overlay")
weight_ratios = normal_counts / del1g_counts

del1g_ke = np.array(del1g_numuCC_df["wc_truth_muonMomentum_3"]) * 1000.0 - 105.658
bin_indices = np.digitize(del1g_ke, bins)
bin_indices = np.clip(bin_indices, 1, len(weight_ratios)) - 1  # convert to 0-based index into weight_ratios
event_weight_ratios = weight_ratios[bin_indices]
del1g_numuCC_df = del1g_numuCC_df.with_columns(
    pl.Series("fix_del1g_weight", event_weight_ratios)
)

plt.hist(np.array(del1g_numuCC_df["wc_truth_muonMomentum_3"])*1000.0 - 105.658, weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"], bins=bins, histtype="step", label="Normal-overlay Reweighted Del1g overlay")
plt.hist(np.array(del1g_numuCC_df["wc_truth_muonMomentum_3"])*1000.0 - 105.658, weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["fix_del1g_weight"] * del1g_numuCC_df["rad_frac_x_eta"], bins=bins, histtype="step", label="Radiative Reweighted Del1g overlay")

plt.legend()
plt.yscale("log")
plt.xlabel("True leading muon kinetic energy [MeV]")
plt.ylabel("Number of events")
plt.title("All true numuCC Del1g Events")
plt.show()

plt.figure(figsize=(10, 6))
bins = np.linspace(-1, 1, 101)
plt.hist(np.array(normal_numuCC_df["wc_true_muon_costheta"]), weights=normal_numuCC_df["wc_net_weight"], bins=bins, histtype="step", label="normal overlay")
plt.hist(np.array(del1g_numuCC_df["wc_true_muon_costheta"]), weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"], bins=bins, histtype="step", label="Normal-overlay Reweighted Del1g overlay")
plt.legend()
plt.xlabel("True muon costheta")
plt.ylabel("Number of events")
plt.title("All true numuCC Events")
plt.show()

plt.figure(figsize=(10, 6))
bins = np.linspace(0, 500, 101)
plt.hist(del1g_numuCC_df["wc_true_leading_shower_energy"], weights=del1g_numuCC_df["wc_net_weight"], bins=bins, histtype="step", label="Del1g overlay")
plt.hist(del1g_numuCC_df["wc_true_leading_shower_energy"], weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"], bins=bins, histtype="step", label="Normal-overlay Reweighted Del1g overlay")
plt.hist(del1g_numuCC_df["wc_true_leading_shower_energy"], weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"] * del1g_numuCC_df["rad_frac_x_eta"], bins=bins, histtype="step", label="Radiative Reweighted Del1g overlay")
plt.legend()
plt.yscale("log")
plt.xlabel("True leading shower energy [MeV]")
plt.ylabel("Number of events")
plt.title("All true numuCC Events")
plt.show()

plt.figure(figsize=(10, 6))
bins = np.linspace(-1, 1, 101)
plt.hist(del1g_numuCC_df["wc_true_leading_shower_costheta"], weights=del1g_numuCC_df["wc_net_weight"], bins=bins, histtype="step", label="Del1g overlay")
plt.hist(del1g_numuCC_df["wc_true_leading_shower_costheta"], weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"], bins=bins, histtype="step", label="Normal-overlay Reweighted Del1g overlay")
plt.hist(del1g_numuCC_df["wc_true_leading_shower_costheta"], weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"] * del1g_numuCC_df["rad_frac_x_eta"], bins=bins, histtype="step", label="Radiative Reweighted Del1g overlay")
plt.legend()
plt.yscale("log")
plt.xlabel("True leading shower costheta")
plt.ylabel("Number of events")
plt.title("All true numuCC Events")
plt.show()


In [ ]:
# Save per-event radiative correction weights for use in other notebooks.
# Must be run after all weight columns have been added to del1g_numuCC_df:
#   rad_frac_x_eta           (cell above)
#   x_eta_uniform_weight     (cell above)
#   fix_del1g_weight         (this cell)
#   wc_muon_gamma_opening_angle (cell above)
# Only events with all weights > 0 are saved; events outside the valid kinematic
# range (e.g. x_eta_uniform_weight=0, fix_del1g_weight=0) are excluded.

weights_to_save = del1g_numuCC_df.filter(
    (pl.col("fix_del1g_weight") > 0) &
    (pl.col("x_eta_uniform_weight") > 0) &
    (pl.col("rad_frac_x_eta") > 0)
).select([
    "run",
    "subrun",
    "event",
    "fix_del1g_weight",
    "x_eta_uniform_weight",
    "rad_frac_x_eta",
    "wc_muon_gamma_opening_angle",
])

weights_to_save.write_parquet(f"{intermediate_files_location}/del1g_rad_correction_weights.parquet")

print(f"Saved del1g_rad_correction_weights.parquet")
print(f"  {len(weights_to_save):,} / {len(del1g_numuCC_df):,} events have valid weights")


# Muon-gamma plots

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(del1g_numuCC_df["rad_corr_x"], weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"] * del1g_numuCC_df["rad_frac_x_eta"], bins=np.linspace(0, 1, 101), histtype="step")
plt.xlabel("x = E_ℓ / E_tree")
plt.ylabel("Number of events")
plt.title("All true numuCC Del1g Events\nRadiative Correction Reweighted")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(del1g_numuCC_df["rad_corr_eta"], weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"] * del1g_numuCC_df["rad_frac_x_eta"], bins=np.linspace(0, 50, 101), histtype="step")
plt.xlabel("eta = Δθ [rad] · E_tree / m_ℓ")
plt.ylabel("Number of events")
plt.title("All true numuCC Del1g Events\nRadiative Correction Reweighted")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(del1g_numuCC_df["rad_frac_x_eta"], weights=del1g_numuCC_df["wc_net_weight"] * del1g_numuCC_df["x_eta_uniform_weight"] * del1g_numuCC_df["fix_del1g_weight"] * del1g_numuCC_df["rad_frac_x_eta"], bins=np.linspace(0, 0.2, 101), histtype="step")
plt.xlabel("rad_frac_x_eta")
plt.ylabel("Number of events")
plt.title("All true numuCC Del1g Events\nRadiative Correction Reweighted")
plt.yscale("log")
plt.show()
